# Expense Tracker — Kaggle Master Training

**Run the notebook from top to bottom.** This notebook trains the complete initial ML pipeline on a Kaggle GPU without modifying Kaggle's system NumPy/Pandas/Torch environment.

The project requires **Python 3.14**, so all ML commands run through the repository's isolated `uv` environment. The notebook does **not** run `pip install -e .` against Kaggle's Python 3.12 kernel.

Models produced by the master pipeline:
- TF-IDF category classifier
- XLM-R Transformer category classifier
- Merchant similarity index
- Duplicate similarity model
- Transaction anomaly model when amount features are available
- Spending forecast model when date + amount features are available


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/Yoge-2004/expense-tracker.git"
BRANCH = "feature/ml-expense-intelligence"
WORK_ROOT = Path("/kaggle/working")
REPO = WORK_ROOT / "expense-tracker"
ML = REPO / "ml"

OUTPUT = WORK_ROOT / "expense-ml-runs"
DATA_CACHE = WORK_ROOT / "expense-ml-data"
HF_CACHE = Path("/kaggle/temp/huggingface")
OUTPUT.mkdir(parents=True, exist_ok=True)
DATA_CACHE.mkdir(parents=True, exist_ok=True)
HF_CACHE.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_DATASETS_CACHE"] = str(HF_CACHE / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE / "transformers")
os.environ["EXPENSE_ML_OUTPUT"] = str(OUTPUT)
os.environ["EXPENSE_ML_DATA_DIR"] = str(DATA_CACHE)
os.environ["EXPENSE_ML_DATA_CACHE"] = str(DATA_CACHE)

def run(*args, cwd=None, env=None):
    merged = os.environ.copy()
    if env:
        merged.update({str(k): str(v) for k, v in env.items()})
    print("$", " ".join(map(str, args)))
    return subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, env=merged, check=True)

if REPO.exists():
    shutil.rmtree(REPO)
run("git", "clone", "--depth", "1", "--branch", BRANCH, "--single-branch", REPO_URL, REPO)
commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Checked-out commit:", commit)
print("ML directory:", ML)

## 1. Verify the Kaggle GPU and load Hugging Face authentication

Enable a Kaggle **GPU accelerator** before running this notebook.

Create a Kaggle Secret named `HF_TOKEN`. It is used only for Hugging Face dataset/model access and is never printed.

In [ ]:
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("nvidia-smi was not found. Enable a Kaggle GPU accelerator before training.")

run("nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader")

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token.strip()
except Exception as exc:
    print("Kaggle Secrets lookup failed:", type(exc).__name__)

if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("HF_TOKEN is required. Add a Kaggle Secret named HF_TOKEN containing a Hugging Face access token.")

print("HF_TOKEN loaded.")

## 2. Create the isolated Python 3.14 training environment

Kaggle's notebook kernel can remain on Python 3.12. The ML project is installed into its own `uv` environment using the Python version declared by the repository.

**Do not run `pip install -e .` and do not replace Kaggle's NumPy/Pandas/Torch packages.**

In [ ]:
if shutil.which("uv") is None:
    run(sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "uv")

UV = shutil.which("uv")
if UV is None:
    candidates = [Path(sys.prefix) / "bin" / "uv", Path.home() / ".local" / "bin" / "uv"]
    UV = next((str(p) for p in candidates if p.exists()), None)
if UV is None:
    raise FileNotFoundError("uv could not be located after installation.")

UV_CACHE = WORK_ROOT / "uv-cache"
UV_CACHE.mkdir(parents=True, exist_ok=True)

run(UV, "python", "install", "3.14")
run(UV, "sync", "--extra", "train", "--extra", "dev", cwd=ML, env={"UV_CACHE_DIR": str(UV_CACHE)})

version = subprocess.check_output([UV, "run", "python", "--version"], cwd=ML, text=True).strip()
print("Project interpreter:", version)
if "3.14" not in version:
    raise RuntimeError(f"uv selected the wrong Python interpreter: {version}")

In [ ]:
preflight = r'''
import sys
import torch
import numpy, pandas, scipy, sklearn, transformers, datasets, accelerate

print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    print("BF16:", torch.cuda.is_bf16_supported())
'''
run(UV, "run", "python", "-c", preflight, cwd=ML, env={"HF_HOME": os.environ["HF_HOME"], "HF_DATASETS_CACHE": os.environ["HF_DATASETS_CACHE"], "TRANSFORMERS_CACHE": os.environ["TRANSFORMERS_CACHE"], "HF_TOKEN": os.environ["HF_TOKEN"]})

## 3. Start from a completely fresh training dataset/cache

The initial run must not silently reuse prepared data or source caches from an earlier experiment. Only disposable Kaggle working directories are cleared.

In [ ]:
for path in (DATA_CACHE, OUTPUT):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
print("Fresh initial-training workspace ready.")

## 4. Kaggle-specific training settings

These settings affect this Kaggle run only. They do not modify the production defaults in the repository.

In [ ]:
cpu_count = os.cpu_count() or 4
KAGGLE_ENV = {
    "HF_TOKEN": os.environ["HF_TOKEN"],
    "HF_HOME": os.environ["HF_HOME"],
    "HF_DATASETS_CACHE": os.environ["HF_DATASETS_CACHE"],
    "TRANSFORMERS_CACHE": os.environ["TRANSFORMERS_CACHE"],
    "EXPENSE_ML_DATA_DIR": str(DATA_CACHE),
    "EXPENSE_ML_OUTPUT": str(OUTPUT),
    "EXPENSE_ML_CPU_THREADS": str(min(cpu_count, 8)),
    "EXPENSE_ML_TORCH_THREADS": str(min(cpu_count, 8)),
    "EXPENSE_ML_DATALOADER_WORKERS": str(min(max(cpu_count // 2, 1), 4)),
    "EXPENSE_ML_BATCH_SIZE": "16",
    "EXPENSE_ML_EVAL_BATCH_SIZE": "64",
    "EXPENSE_ML_MIXED_PRECISION": "auto",
    "EXPENSE_ML_NO_PIN_MEMORY": "0",
    "EXPENSE_ML_FORCE_FETCH": "1",
}
print(json.dumps({k: v for k, v in KAGGLE_ENV.items() if k != "HF_TOKEN"}, indent=2))

## 5. Run the complete master pipeline

This is the only training cell needed for the initial model. It calls `expense_ml.master_pipeline` inside the isolated Python 3.14 environment. Because the prepared dataset does not exist yet, the master pipeline will fetch and normalize the datasets from `config/datasets.yaml`, then train and evaluate the complete model suite.

In [ ]:
prepared = DATA_CACHE / "transactions.parquet"
config = ML / "config" / "datasets.yaml"
run(
    UV, "run", "python", "-m", "expense_ml.master_pipeline",
    "--config", str(config),
    "--prepared", str(prepared),
    "--output", str(OUTPUT),
    cwd=ML,
    env=KAGGLE_ENV,
)

## 6. Inspect the completed run

The notebook stops with an error if the master pipeline did not finish successfully.

In [ ]:
run_dirs = sorted(path for path in OUTPUT.iterdir() if path.is_dir() and (path / "manifest.json").exists())
if not run_dirs:
    raise RuntimeError("No master-training run with manifest.json was produced.")

RUN_DIR = run_dirs[-1]
MANIFEST = json.loads((RUN_DIR / "manifest.json").read_text(encoding="utf-8"))
print("Run:", RUN_DIR.name)
print("Status:", MANIFEST.get("status"))
print("Pipeline:", MANIFEST.get("pipeline_version"))
print("\nSelected model:")
print(json.dumps(MANIFEST.get("selected_model"), indent=2))
print("\nFinal test:")
print(json.dumps(MANIFEST.get("test_evaluation"), indent=2))
if MANIFEST.get("india_holdout_evaluation"):
    print("\nIndia holdout:")
    print(json.dumps(MANIFEST["india_holdout_evaluation"], indent=2))
print("\nModel statuses:")
for name, details in MANIFEST.get("models", {}).items():
    print(f"  {name}: {details.get('status')}")

if MANIFEST.get("status") != "completed":
    raise RuntimeError("Master pipeline did not finish with status=completed.")

In [ ]:
reports = [
    "reports/dataset_summary.json",
    "reports/dataset_quality.json",
    "reports/split_summary.json",
    "reports/training_sampling.json",
    "reports/model_selection_validation.json",
    "reports/model_comparison.json",
    "reports/category_test.json",
    "reports/category_test_country_metrics.json",
    "reports/category_india_holdout.json",
    "reports/duplicate_candidates.json",
    "reports/anomaly_report.json",
    "reports/spending_forecast.json",
]

for relative in reports:
    path = RUN_DIR / relative
    if path.exists():
        print(f"\n===== {relative} =====")
        print(path.read_text(encoding="utf-8")[:12000])

## 7. Package the initial model artifacts

The archive contains the complete master run: trained models, manifests, reports, and figures. Raw datasets and Hugging Face caches are not included.

In [ ]:
archive_base = WORK_ROOT / f"expense-tracker-ml-initial-{RUN_DIR.name}"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=RUN_DIR))
print("Kaggle artifact:", archive_path)
print("Artifact size MiB:", round(archive_path.stat().st_size / 1024**2, 2))
print("Run directory:", RUN_DIR)